# 6. Cotunnelling and second-order methods

**Learning goals.** In this tutorial you will:

- understand why sequential tunnelling predicts (almost) no current inside a Coulomb diamond, and what fills it in;
- run the RTD and 2vN second-order approaches and check their convergence;
- read cotunnelling off a stability diagram, and separate real features from perturbative artifacts;
- see a first-order calculation overestimate the efficiency of a quantum-dot heat engine by a factor of two; and
- use the RTD-specific controls: `off_diag_corrections`, complex tunnelling amplitudes, and many-body input.

This is the capstone of the learning path. It assumes the Anderson model (Tutorial 2), sweeps (Tutorial 3), the approach comparison (Tutorial 4), and the energy and heat currents (Tutorial 5).

## Why first order is not enough

A first-order rate is a *real* transition: an electron moves between a lead and the dot, and the dot changes charge. Inside a Coulomb diamond every such transition costs more energy than the bias and temperature can supply, so first-order theory predicts a current that is only thermally activated — exponentially small.

At second order in $\Gamma$ a pair of tunnelling events happens coherently through a *virtual* intermediate state. The intermediate state may violate energy conservation because it is never occupied; only the initial and final states must balance. This is **cotunnelling**:

- **elastic** cotunnelling leaves the dot in its initial state and gives a current at arbitrarily small bias;
- **inelastic** cotunnelling leaves the dot in a different state of the same charge, and switches on when $|V|$ exceeds the excitation energy.

QmeQ implements two second-order approaches:

- **RTD** (real-time diagrammatics) is the second-order extension of the Pauli equation: it works with a diagonal density matrix and is exact to order $\Gamma^2$. It therefore cannot handle degenerate or nearly degenerate states ($\Delta E\lesssim\Gamma$) that couple to the same lead — the situation Tutorial 4 was about. Spin degeneracy is fine, because different spins couple to different lead channels in QmeQ.
- **2vN** (second-order von Neumann) expands in the number of particle-hole excitations rather than in $\Gamma$, and solves an integral equation on an energy grid.

Both remain perturbative: they require $\Gamma\ll T$ and do not describe the Kondo effect.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
import qmeq
from scipy.optimize import brentq

print(qmeq.get_backend_status())

## Practical requirements

The RTD kernel is built for `itype=1`, which treats the lead integrals in the wide-band limit, so `dband` should be much larger than every other energy in the problem. RTD also supports only `indexing='charge'` (the default here) and neither the matrix-free solver (`mfreeq=True`) nor `symmetry=True`.

The 2vN kernel needs an explicit energy grid: `kpnt` points spanning the band, and `solve(niter=...)` iterations of the integral equation. Its bandwidth has to stay moderate, because the grid must resolve the temperature across the whole band.

We first put all four approaches on the same footing: an Anderson dot in the middle of its Coulomb diamond, with the bias well below the addition energy $U/2$.

In [ ]:
U = 20.0
temperature = 1.0
gamma = 0.5
tunnel_amplitude = np.sqrt(gamma / (2 * np.pi))
blockade_gate = -U / 2
blockade_bias = 4.0  # far below the addition energy U/2 = 10

def make_anderson(kerntype, gate=blockade_gate, bias=blockade_bias,
                  bandwidth=60.0, zeeman=0.0, charging=U, **kwargs):
    return qmeq.Builder(
        nsingle=2,
        hsingle={(0, 0): gate - zeeman / 2, (1, 1): gate + zeeman / 2},
        coulomb={(0, 1, 1, 0): charging},
        nleads=4,
        tleads={(0, 0): tunnel_amplitude, (1, 0): tunnel_amplitude,
                (2, 1): tunnel_amplitude, (3, 1): tunnel_amplitude},
        mulst={0: bias / 2, 1: -bias / 2, 2: bias / 2, 3: -bias / 2},
        tlst={0: temperature, 1: temperature,
              2: temperature, 3: temperature},
        dband=bandwidth,
        kerntype=kerntype,
        **kwargs,
    )

blockade_currents = {}
for kerntype in ["Pauli", "1vN"]:
    system = make_anderson(kerntype)
    system.solve()
    blockade_currents[kerntype] = system.current[0] + system.current[2]

# RTD wants a wide band and itype=1.
rtd = make_anderson("RTD", bandwidth=5.0e5, itype=1)
rtd.solve()
blockade_currents["RTD"] = rtd.current[0] + rtd.current[2]

# 2vN keeps a moderate band and resolves it with an energy grid.
second_vn = make_anderson("2vN", kpnt=2**10)
second_vn.solve(niter=5)
blockade_currents["2vN"] = second_vn.current[0] + second_vn.current[2]

for name, value in blockade_currents.items():
    print(f"{name:6s} I_L/Gamma = {value / gamma:.6e}")

first_order = blockade_currents["Pauli"]
assert np.isclose(blockade_currents["1vN"], first_order, rtol=1e-6)
# Both second-order approaches find a current far above the first-order one,
# and they agree with each other to a few percent.
assert blockade_currents["RTD"] > 50 * first_order
assert blockade_currents["2vN"] > 50 * first_order
assert np.isclose(blockade_currents["RTD"], blockade_currents["2vN"], rtol=0.05)

print(f"\nsecond order / first order: "
      f"{blockade_currents['RTD'] / first_order:.0f}x")
print(f"RTD vs 2vN disagreement: "
      f"{abs(blockade_currents['RTD'] / blockade_currents['2vN'] - 1):.1%}")

Inside the diamond the first-order approaches are wrong by a factor of about seventy, and the two second-order approaches — built on different expansions — agree with each other to a few percent. That mutual agreement is the strongest evidence available here that the cotunnelling current is real physics rather than an artifact of one scheme.

## Is a feature real? Test its scaling in $\Gamma$

A perturbative result is trustworthy only where the next order is negligible. Cotunnelling is second order, so a genuine cotunnelling current scales as $\Gamma^2$. If a feature instead scales as $\Gamma^3$ or faster, it is dominated by the terms the approach has *dropped*, and it should not be interpreted physically. This is the single most useful check when using RTD or 2vN at the edge of their validity.

In [ ]:
scaled_gammas = np.array([0.125, 0.25, 0.5, 1.0, 2.0])
scaled_currents = {"RTD": np.empty_like(scaled_gammas),
                   "Pauli": np.empty_like(scaled_gammas)}

for name in scaled_currents:
    for index, scaled_gamma in enumerate(scaled_gammas):
        amplitude = np.sqrt(scaled_gamma / (2 * np.pi))
        system = make_anderson(name, bandwidth=5.0e5, itype=1)
        system.change(tleads={(0, 0): amplitude, (1, 0): amplitude,
                              (2, 1): amplitude, (3, 1): amplitude})
        system.solve()
        scaled_currents[name][index] = system.current[0] + system.current[2]

def local_exponent(values):
    """Effective power of Gamma between successive couplings."""
    return np.diff(np.log(values)) / np.diff(np.log(scaled_gammas))

print(f"{'Gamma':>7} {'I (RTD)':>13} {'I (Pauli)':>13}"
      f" {'d ln I / d ln Gamma':>21}")
rtd_exponents = local_exponent(scaled_currents["RTD"])
pauli_exponents = local_exponent(scaled_currents["Pauli"])
for index, scaled_gamma in enumerate(scaled_gammas):
    exponents = ("" if index == 0
                 else f"RTD {rtd_exponents[index - 1]:.3f}, "
                      f"Pauli {pauli_exponents[index - 1]:.3f}")
    print(f"{scaled_gamma:7.3f} {scaled_currents['RTD'][index]:13.6e}"
          f" {scaled_currents['Pauli'][index]:13.6e} {exponents:>21}")

assert np.all((rtd_exponents > 1.9) & (rtd_exponents < 2.01))
assert np.allclose(pauli_exponents, 1.0, atol=1e-6)
print("\nRTD current scales as Gamma^2, the sequential current as Gamma^1")

The exponents come out just under two and approach it as $\Gamma$ grows, because the total current also contains the thermally activated *first*-order piece; that contribution is relatively larger at small $\Gamma$, and the Pauli column isolates it with its exact exponent of one.

Note what this test does *not* say: the largest coupling here, $\Gamma=2T$, violates $\Gamma\ll T$, and the current is still essentially $\propto\Gamma^2$ because that is the order the kernel is built at. The scaling test detects features contaminated by *neglected* orders; it cannot tell you that a perturbative expansion has converged. For that, compare with an approach based on a different expansion, as above.

## Cotunnelling in a stability diagram

Add a Zeeman splitting $E_Z$ so the dot has an excited state at fixed charge, and widen the diamond ($U=60$, with RTD's wide band) so there is room to see structure inside it. Two features should appear where first order gave nothing: a small bias-independent *elastic* cotunnelling conductance around $V=0$, and a step up when $|V|$ crosses $E_Z$ and *inelastic* cotunnelling can leave the dot in its excited spin state.

In [ ]:
map_U = 60.0
zeeman = 10.0
map_gate = -map_U / 2
map_gates = np.linspace(-1.4 * map_U, 0.4 * map_U, 46)
map_biases = np.linspace(-1.4 * map_U, 1.4 * map_U, 57)

def make_split_dot(kerntype, gate=map_gate):
    return make_anderson(kerntype, gate=gate, bias=0.0, bandwidth=5.0e5,
                         zeeman=zeeman, charging=map_U, itype=1)

cotunnelling = make_split_dot("RTD", gate=map_gates[0])
current_map = np.empty((len(map_biases), len(map_gates)))

for gate_index, gate in enumerate(map_gates):
    cotunnelling.change(hsingle={(0, 0): gate - zeeman / 2,
                                 (1, 1): gate + zeeman / 2})
    cotunnelling.solve(masterq=False)
    for bias_index, bias in enumerate(map_biases):
        cotunnelling.change(mulst={0: bias / 2, 1: -bias / 2,
                                   2: bias / 2, 3: -bias / 2})
        cotunnelling.solve(qdq=False)
        current_map[bias_index, gate_index] = (cotunnelling.current[0]
                                               + cotunnelling.current[2])

conductance_map = np.gradient(current_map, map_biases, axis=0)
assert np.all(conductance_map > 0.0)  # no negative differential conductance here

In [ ]:
# A fine bias cut through the middle of the diamond, for both orders.
cut_biases = np.linspace(-3 * zeeman, 3 * zeeman, 121)
cut_conductance = {}

for kerntype in ["RTD", "Pauli"]:
    system = make_split_dot(kerntype)
    cut_current = np.empty_like(cut_biases)
    for index, bias in enumerate(cut_biases):
        system.change(mulst={0: bias / 2, 1: -bias / 2,
                             2: bias / 2, 3: -bias / 2})
        system.solve()
        cut_current[index] = system.current[0] + system.current[2]
    cut_conductance[kerntype] = np.gradient(cut_current, cut_biases)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4))
image = axes[0].imshow(
    conductance_map, origin="lower", aspect="auto", cmap="magma",
    norm=colors.LogNorm(vmin=conductance_map.min(), vmax=conductance_map.max()),
    extent=[map_gates[0] / map_U, map_gates[-1] / map_U,
            map_biases[0] / map_U, map_biases[-1] / map_U],
)
fig.colorbar(image, ax=axes[0], label="$\\partial I_L/\\partial V$")
axes[0].axhline(zeeman / map_U, color="w", ls=":", lw=0.8)
axes[0].axhline(-zeeman / map_U, color="w", ls=":", lw=0.8)
axes[0].set(xlabel="$\\varepsilon/U$", ylabel="$V/U$",
            title="RTD conductance (log scale)")

for kerntype, style in [("RTD", "k-"), ("Pauli", "C3--")]:
    axes[1].semilogy(cut_biases / zeeman, cut_conductance[kerntype], style,
                     label=kerntype)
for threshold in (-1.0, 1.0):
    axes[1].axvline(threshold, color="C1", ls=":", lw=1)
axes[1].set(xlabel="$V/E_Z$", ylabel="$\\partial I_L/\\partial V$",
            title="Cut through the diamond centre")
axes[1].legend()
fig.tight_layout()

plateau = abs(cut_biases) < 0.4 * zeeman
inelastic = (abs(cut_biases) > 1.2 * zeeman) & (abs(cut_biases) < 2.0 * zeeman)
step = cut_conductance["RTD"][inelastic].mean() / cut_conductance["RTD"][plateau].mean()
zero_bias = np.argmin(abs(cut_biases))

print(f"elastic plateau, RTD:   {cut_conductance['RTD'][zero_bias]:.3e}")
print(f"elastic plateau, Pauli: {cut_conductance['Pauli'][zero_bias]:.3e}")
print(f"inelastic step across |V| = E_Z: {step:.1f}x")
assert step > 2.0
assert cut_conductance["RTD"][zero_bias] > 1e6 * cut_conductance["Pauli"][zero_bias]

The diamond is no longer empty: on a logarithmic colour scale there is weak but finite conductance everywhere inside it, with the dotted lines marking $|V|=E_Z$. The cut makes the mechanism explicit — a plateau of elastic cotunnelling around zero bias, then a step of about five times that value once $|V|>E_Z$ opens the inelastic channel. The Pauli curve, for the same model, is eleven orders of magnitude below the plateau and rises only through thermal activation.

In experiments these inelastic cotunnelling steps are the standard way to measure excitation energies deep inside a diamond, where sequential transport is blocked.

## Convergence of the 2vN solution

The 2vN approach needs two numerical controls, and both must be checked:

- `niter` — iterations of the integral equation. `system.iters[i]` stores the solution after each one, so the sequence of currents shows directly whether it has converged.
- `kpnt` — points in the energy grid. Too few and the current depends on the discretization.

We look at a *resonant* point, where the iteration actually has work to do; at the blockade point used above it converges on the first pass.

In [ ]:
resonant = make_anderson("2vN", gate=0.0, bias=blockade_bias, kpnt=2**10)
resonant.solve(niter=6)

previous = None
for iteration in range(resonant.niter + 1):
    value = (resonant.iters[iteration].current[0]
             + resonant.iters[iteration].current[2])
    change = "" if previous is None else f"  change = {abs(value - previous):.2e}"
    print(f"iteration {iteration}: I_L = {value:.8f}{change}")
    previous = value

converged = resonant.current[0] + resonant.current[2]
last_change = abs(converged - (resonant.iters[resonant.niter - 1].current[0]
                               + resonant.iters[resonant.niter - 1].current[2]))
assert last_change < 1e-5 * abs(converged)

print()
for exponent in [9, 10, 11]:
    grid = make_anderson("2vN", gate=0.0, bias=blockade_bias, kpnt=2**exponent)
    grid.solve(niter=6)
    value = grid.current[0] + grid.current[2]
    print(f"kpnt = 2**{exponent}: I_L = {value:.8f}")
    assert np.isclose(value, converged, rtol=1e-3)

print("\n2vN converged in both the iteration and the energy grid")

Both controls are converged here, and neither convergence says anything about the *physical* accuracy of the second-order expansion — a converged 2vN result at $\Gamma\sim T$ is a precisely computed approximation, not a precise answer.

## A quantum-dot heat engine

Tutorial 5 showed that a first-order dot is a perfect energy filter: the heat current is locked to the particle current, so the thermovoltage — the bias that stalls the current — follows

$$V_\mathrm{th}=-\frac{2\Delta\,\Delta T}{2T+\Delta T},$$

with $\Delta$ the addition energy closest to the chemical potential — valid where one transition dominates, which excludes the neighbourhood of $\varepsilon=-U/2$, where the two addition energies contribute equally and symmetry forces $V_\mathrm{th}=0$. Cotunnelling breaks that lock: it transfers energy without a real charge transition, so it can carry heat while the particle current is zero. The consequences for a thermoelectric device are dramatic.

We keep the dot at $T_\mathrm{hot}=T+\Delta T$ on the left channels and $T$ on the right, and first compute $V_\mathrm{th}$ from $I(V_\mathrm{th})=0$.

In [ ]:
engine_U = 500.0
engine_temperature, delta_temperature = 10.0, 10.0
engine_gamma = 1.0
engine_amplitude = np.sqrt(engine_gamma / (2 * np.pi))
carnot_efficiency = delta_temperature / (engine_temperature + delta_temperature)

def make_engine(kerntype):
    hot = engine_temperature + delta_temperature
    return qmeq.Builder(
        nsingle=2,
        hsingle={(0, 0): 0.0, (1, 1): 0.0},
        coulomb={(0, 1, 1, 0): engine_U},
        nleads=4,
        tleads={(0, 0): engine_amplitude, (1, 0): engine_amplitude,
                (2, 1): engine_amplitude, (3, 1): engine_amplitude},
        mulst={0: 0.0, 1: 0.0, 2: 0.0, 3: 0.0},
        tlst={0: hot, 1: engine_temperature, 2: hot, 3: engine_temperature},
        dband=5.0e5,
        kerntype=kerntype,
        itype=1,
    )

def operating_point(system, gate, voltage):
    """Particle and heat current drawn from the hot (left) channels."""
    system.change(hsingle={(0, 0): gate, (1, 1): gate},
                  mulst={0: voltage / 2, 1: -voltage / 2,
                         2: voltage / 2, 3: -voltage / 2})
    system.solve()
    return (system.current[0] + system.current[2],
            system.heat_current[0] + system.heat_current[2])

engine_gates = np.linspace(-1.5 * engine_U, 0.5 * engine_U, 41)
thermovoltage = {}
engines = {name: make_engine(name) for name in ["Pauli", "RTD"]}

for name, system in engines.items():
    thermovoltage[name] = np.array([
        brentq(lambda voltage: operating_point(system, gate, voltage)[0],
               -400.0, 400.0, xtol=1e-8)
        for gate in engine_gates
    ])

def tight_coupling_thermovoltage(gate):
    addition_energies = np.array([gate, gate + engine_U])
    nearest = addition_energies[np.argmin(abs(addition_energies))]
    return (-2 * nearest * delta_temperature
            / (2 * engine_temperature + delta_temperature))

expected = np.array([tight_coupling_thermovoltage(g) for g in engine_gates])
# The single-transition formula applies where one addition energy dominates,
# i.e. away from the particle-hole symmetric point eps = -U/2.
dominant = abs(engine_gates + engine_U / 2) > 0.2 * engine_U
assert np.allclose(thermovoltage["Pauli"][dominant], expected[dominant],
                   rtol=2e-2, atol=1.0)

print("gate/U   V_th (Pauli)   tight coupling   V_th (RTD)")
for index in range(0, len(engine_gates), 8):
    print(f"{engine_gates[index] / engine_U:6.2f} "
          f"{thermovoltage['Pauli'][index]:14.2f} {expected[index]:16.2f} "
          f"{thermovoltage['RTD'][index]:12.2f}")

The first-order thermovoltage grows without bound as the dot is pushed deeper into blockade, exactly as tight coupling demands. The RTD thermovoltage collapses: cotunnelling provides a channel that carries current at much smaller bias, so far less voltage is needed to stall the net flow. Deep in the diamond the two predictions differ by more than an order of magnitude.

Tutorial 5 checked that a stalled first-order device transports *nothing*: zero particle current forced zero heat current. That is the assumption second order breaks, and it can be tested directly at the stall point.

In [ ]:
stall_gate = -1.1 * engine_U
for name, system in engines.items():
    stall_voltage = brentq(
        lambda voltage: operating_point(system, stall_gate, voltage)[0],
        -400.0, 400.0, xtol=1e-10,
    )
    particle, heat = operating_point(system, stall_gate, stall_voltage)
    print(f"{name:6s} V_th = {stall_voltage:8.3f}: I = {particle:.3e}, "
          f"J^Q_hot = {heat:.3e}")
    if name == "Pauli":
        assert abs(heat) < 1e-8  # tight coupling: no particles, no heat
    else:
        assert heat > 1e-2  # cotunnelling carries heat through a stalled dot

At the RTD stall point the dot passes no net charge yet still conducts heat out of the hot reservoir — a thermal conductance that first order cannot produce at all. That is the mechanism behind everything that follows.

Now close the circuit. Connecting a load resistance $R$ instead of a voltage source fixes the operating point through Kirchhoff's laws,

$$I_\mathrm{dot}(V)+\frac{V}{R}=0,$$

which we solve for $V$ at each gate. The generated power is $P=-VI$, and the efficiency is $\eta=P/J^Q_\mathrm{hot}$, bounded by the Carnot value $\eta_C=\Delta T/(T+\Delta T)$.

In [ ]:
load_resistance = 5 * 243.0  # 5 MOhm in units of hbar/e^2

power, efficiency = {}, {}
for name, system in engines.items():
    generated = np.empty_like(engine_gates)
    ratio = np.full_like(engine_gates, np.nan)
    for index, gate in enumerate(engine_gates):
        voltage = brentq(
            lambda v: operating_point(system, gate, v)[0] + v / load_resistance,
            -400.0, 400.0, xtol=1e-8,
        )
        particle, heat = operating_point(system, gate, voltage)
        generated[index] = -voltage * particle
        if heat > 0.0:
            ratio[index] = generated[index] / heat
    power[name] = generated
    efficiency[name] = ratio

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
for name, style in [("RTD", "k-"), ("Pauli", "C3--")]:
    axes[0].plot(engine_gates / engine_U, power[name], style, label=name)
    axes[1].plot(engine_gates / engine_U,
                 efficiency[name] / carnot_efficiency, style, label=name)
axes[0].set(xlabel="$\\varepsilon/U$", ylabel="$P$",
            title=f"Power into a {load_resistance / 243:.0f} M$\\Omega$ load")
axes[1].set(xlabel="$\\varepsilon/U$", ylabel="$\\eta/\\eta_C$",
            title="Efficiency relative to Carnot")
for axis in axes:
    axis.legend()
fig.tight_layout()

for name in ["Pauli", "RTD"]:
    print(f"{name:6s} max power = {np.nanmax(power[name]):.4f}, "
          f"max eta/eta_C = {np.nanmax(efficiency[name]) / carnot_efficiency:.3f}")

# The second law must hold in both approaches, and first order must not be
# trusted for the efficiency: it overestimates it substantially here.
for name in ["Pauli", "RTD"]:
    assert np.nanmax(efficiency[name]) <= carnot_efficiency + 1e-9
    assert np.nanmin(power[name]) > -1e-12
assert np.nanmax(efficiency["Pauli"]) > 1.3 * np.nanmax(efficiency["RTD"])

The generated power differs modestly between the two approaches, but the efficiency does not: the first-order calculation reaches $0.93\,\eta_C$ while RTD gives $0.66\,\eta_C$. The reason is the tight coupling of Tutorial 5. In first order, every transported electron carries exactly the addition energy, which makes the dot a reversible energy filter; cotunnelling adds heat flow that generates no power, and the efficiency drops. A device design based on the first-order number would be wrong by a wide margin — this is the practical reason second-order methods exist.

## RTD controls

### Off-diagonal corrections

RTD solves for a diagonal density matrix, but a consistent expansion to order $\Gamma^2$ must still include a correction term generated by the first-order *off-diagonal* elements. QmeQ includes it by default; `system.off_diag_corrections = False` switches it off. The term vanishes by selection rules in models where each single-particle state couples to one lead only (the Anderson model above), but it matters as soon as several dot states share a lead — for example two levels, split by $\Delta\varepsilon$, both coupled to both leads.

In [ ]:
level_splitting, shared_U = 20.0, 1.0e5
shared_temperature, shared_bias = 10.0, 30.0
shared_amplitude = np.sqrt(1.0 / (2 * np.pi))

shared = qmeq.Builder(
    nsingle=2,
    hsingle={(0, 0): 0.0, (1, 1): level_splitting},
    coulomb={(0, 1, 1, 0): shared_U},
    nleads=2,
    # Both levels couple to both leads.
    tleads={(0, 0): shared_amplitude, (1, 0): shared_amplitude,
            (0, 1): shared_amplitude, (1, 1): shared_amplitude},
    mulst={0: -shared_bias / 2, 1: shared_bias / 2},
    tlst={0: shared_temperature, 1: shared_temperature},
    dband=1.0e6,
    kerntype="RTD",
    itype=1,
)

shared_gates = np.linspace(-0.2 * shared_U, 1.2 * shared_U, 71)
with_corrections = np.empty_like(shared_gates)
without_corrections = np.empty_like(shared_gates)

for index, gate in enumerate(shared_gates):
    shared.change(hsingle={(0, 0): -gate, (1, 1): level_splitting - gate})
    shared.off_diag_corrections = True
    shared.solve()
    with_corrections[index] = shared.current[0]
    shared.off_diag_corrections = False
    shared.solve()
    without_corrections[index] = shared.current[0]
shared.off_diag_corrections = True

relative_error = abs((without_corrections - with_corrections) / with_corrections)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
axes[0].semilogy(shared_gates / shared_U, abs(with_corrections), "C3",
                 label="with corrections")
axes[0].semilogy(shared_gates / shared_U, abs(without_corrections), "C0--",
                 label="without")
axes[0].set(xlabel="$V_g/U$", ylabel="$|I_L|$", title="Current")
axes[0].legend()
axes[1].semilogy(shared_gates / shared_U, relative_error, "k")
axes[1].set(xlabel="$V_g/U$", ylabel="relative error", title="Effect of dropping the corrections")
fig.tight_layout()

resonance = np.argmin(abs(shared_gates))
blockade = (shared_gates > 0.1 * shared_U) & (shared_gates < 0.9 * shared_U)

print(f"relative change on resonance:        {relative_error[resonance]:.1%}")
print(f"median relative change in blockade:  "
      f"{np.median(relative_error[blockade]):.1%}")
print(f"sign changes of the current:         "
      f"{np.sum(np.diff(np.sign(with_corrections)) != 0)}")
assert relative_error[resonance] > 0.02
assert np.median(relative_error[blockade]) > 0.5

The correction matters even though the two levels are separated by $\Delta\varepsilon=2T$, far more than $\Gamma$. On resonance dropping it shifts the current by a few percent. Inside the blockade region, where the current is some five orders of magnitude smaller, the correction is comparable to the *entire* current: the median change is above 100%, and it can flip the sign. (The relative measure diverges wherever the current passes through zero, so read the median rather than the maximum.)

Turning the corrections off saves a little memory and no appreciable time, so leave them on unless you are deliberately studying their size.

### Complex tunnelling amplitudes

RTD accepts complex amplitudes. A global phase on every amplitude is unobservable, which makes a useful invariance test; the same is not true of *relative* phases, which change the interference between paths and therefore the current.

One limitation: when the products of amplitudes make the energy kernels complex, RTD cannot evaluate the energy current and says so with a warning. The particle current remains available.

In [ ]:
phase = 0.7  # any fixed value; a global phase must not matter

rotated = make_engine("RTD")
rotated_amplitude = engine_amplitude * np.exp(1j * phase)
rotated.change(tleads={(0, 0): rotated_amplitude, (1, 0): rotated_amplitude,
                       (2, 1): rotated_amplitude, (3, 1): rotated_amplitude})

reference_current, reference_heat = operating_point(engines["RTD"], -100.0, 0.0)
rotated_current, rotated_heat = operating_point(rotated, -100.0, 0.0)

print(f"phase 0:     I = {reference_current:.12e}, J^Q = {reference_heat:.12e}")
print(f"phase {phase}:   I = {rotated_current:.12e}, J^Q = {rotated_heat:.12e}")
assert np.isclose(rotated_current, reference_current, rtol=1e-12)
assert np.isclose(rotated_heat, reference_heat, rtol=1e-12)
print("\ncurrents are invariant under a global phase, as they must be")

### Many-body input and the energy current

`qmeq.BuilderManyBody` takes many-body energies and tunnelling matrix elements directly, which is how you feed QmeQ a dot Hamiltonian it cannot build itself. There is a catch specific to RTD: two of the three kernels that make up the energy current depend on the *single-particle* amplitudes `tleads`, not only on the many-body matrix elements `Tba`. With many-body input those amplitudes are unknown, so the two terms are silently dropped — QmeQ warns about it — and the energy and heat currents are wrong.

The remedy is to supply `nsingle` and `tleads_array` (an `nleads` &times; `nsingle` array) alongside the many-body input. The particle current is unaffected either way. As above, this only matters when a single-particle state connects to more than one lead, so we use exactly such a model: two hybridized levels, both coupled to both leads.

In [ ]:
hybridization = 5.0
many_body_U = 500.0
amplitude = 1.0 / np.sqrt(2 * np.pi)
single_particle_amplitudes = {(0, 0): amplitude, (0, 1): amplitude / 10,
                              (1, 1): amplitude, (1, 0): amplitude / 10}
lead_temperatures = [20.0, 10.0]

reference = qmeq.Builder(
    nsingle=2, nleads=2,
    hsingle={(0, 0): 0.0, (1, 1): 0.0, (0, 1): hybridization},
    coulomb={(0, 1, 1, 0): many_body_U},
    tleads=single_particle_amplitudes,
    mulst=[0.0, 0.0], tlst=lead_temperatures, dband=5.0e5,
    kerntype="RTD", itype=1,
)
reference.solve()
print("many-body energies:", np.round(reference.Ea, 4))

amplitude_array = np.zeros((2, 2), dtype=complex)
for (lead, level), value in single_particle_amplitudes.items():
    amplitude_array[lead, level] = value

def make_many_body(with_single_particle):
    # Build with 'pyRTD' and switch: passing kerntype='RTD' to BuilderManyBody
    # does not pick up amplitudes assigned after construction.
    system = qmeq.BuilderManyBody(
        Ea=reference.qd.Ea, Na=[0, 1, 1, 2], Tba=reference.Tba,
        mulst=[0.0, 0.0], tlst=lead_temperatures, dband=5.0e5,
        kerntype="pyRTD", itype=1,
    )
    system.kerntype = "RTD"
    if with_single_particle:
        system.nsingle = 2
        system.tleads_array = amplitude_array
    return system

In [ ]:
with_amplitudes = make_many_body(True)
without_amplitudes = make_many_body(False)

gates = np.linspace(-200.0, many_body_U + 200.0, 61)
errors = {"with tleads_array": [], "without": []}

for gate in gates:
    reference.change(hsingle={(0, 0): -gate, (1, 1): -gate,
                              (0, 1): hybridization})
    reference.solve()
    exact = reference.energy_current[0]
    for label, system in [("with tleads_array", with_amplitudes),
                          ("without", without_amplitudes)]:
        # Feed the same many-body spectrum without re-diagonalizing.
        system.Ea = np.array([0.0, -gate - hybridization,
                              -gate + hybridization, many_body_U - 2 * gate])
        system.si.states_changed = True
        system.solve(qdq=False, rotateq=False)
        errors[label].append(abs((system.energy_current[0] - exact) / exact))

fig, axis = plt.subplots(figsize=(6, 3.8))
axis.semilogy(gates / many_body_U, errors["without"], "C0--",
              label="many-body input only")
axis.semilogy(gates / many_body_U, np.maximum(errors["with tleads_array"], 1e-17),
              "k", label="with tleads_array")
axis.set(xlabel="$V_g/U$", ylabel="relative error in $J_L$",
         title="Energy current from many-body input")
axis.legend()
fig.tight_layout()

print(f"largest error without single-particle amplitudes: "
      f"{max(errors['without']):.2e}")
print(f"largest error with them:                          "
      f"{max(errors['with tleads_array']):.2e}")
assert max(errors["with tleads_array"]) < 1e-10
assert max(errors["without"]) > 1e-4

Supplying the single-particle amplitudes reproduces the reference energy current to machine precision; omitting them leaves errors of a percent or more, which grow in the blockade regions where the neglected terms dominate. The particle current, which does not use those kernels, is correct in both cases.

## Validity, in one place

| requirement | why |
| --- | --- |
| $\Gamma\ll T$ | both approaches are perturbative; nothing here describes Kondo physics |
| features scale as $\Gamma^2$ | otherwise the result is dominated by neglected orders |
| RTD: `itype=1`, `dband` $\gg$ all energies | the kernel is derived in the wide-band limit |
| RTD: no near-degenerate states on one lead | RTD propagates a diagonal density matrix (Tutorial 4) |
| RTD: `indexing='charge'`, no `mfreeq`/`symmetry` | unsupported combinations |
| 2vN: converged in `niter` and `kpnt` | numerical controls, checked separately from physics |
| 2vN: moderate `dband` | the energy grid must resolve $T$ across the band |
| agreement of RTD and 2vN | different expansions agreeing is real evidence; either alone is not |

## Where to go next

You now have the whole path: a single level and the rate equation, interactions and blockade, sweeps and stability diagrams, coherence and the choice of first-order kernel, energy and heat, and second-order transport. The reference notebooks in `examples/appendix/`, on state types and on symmetries, cover the remaining input conventions and the `indexing` options that make larger models tractable, and `examples/scripts/` holds short programs to copy from. QmeQ also provides electron-phonon variants of the first-order approaches (`BuilderElPh`), which follow the same pattern as the calculations here with a phonon bath added.

## Exercises

1. Move the blockade comparison to $V=8$, just below the addition energy $U/2=10$. Does the ratio between the second-order and first-order currents grow or shrink, and why?
2. Repeat the $\Gamma$-scaling test at $\Gamma=4T$. The scaling stays quadratic — explain why that is *not* evidence that the calculation is valid there.
3. Halve `zeeman` in the stability diagram. Predict where the inelastic step moves before running it.
4. In the heat engine, sweep `load_resistance` at fixed gate and locate the maximum-power point. How far from Carnot is the efficiency there in each approach?
5. Give the amplitudes *relative* phases (`(0, 0)` real, `(1, 0)` rotated) in the shared-lead model of the off-diagonal-correction section. The particle current now changes with the phase — that is physics, not a bug — and QmeQ warns that the energy current cannot be evaluated.